# Notebook 6: Knowledge-initialised code embeddings and knowledge-conditioned readout

Planned comparison of the report (Section "Knowledge injection"). Each configuration changes **one** factor
relative to the Notebook 2 backbone; splits, labels, optimiser and early stopping are unchanged.

| config | input embeddings $Z$ | readout | question |
|---|---|---|---|
| `baseline` | random ($\mathcal N(0, 0.01^2)$) | linear $W_{out}$ | reference = Notebook 2 |
| `desc` | $z_i = a\,W e_i + \delta_i$, $e_i$ = encoder(code description) | linear | does an LLM text prior help rare labels? |
| `rag` | same, with retrieved knowledge in the diagnosis documents | linear | does retrieved knowledge add to descriptions? |
| `rag_shrink` | `rag` + penalty $\lambda_0 \sum_i \|\delta_i\|^2/(1+n_i)$ | linear | should rare codes stay near their prior? |
| `readout` | random | $w_r + U e_r$ (shared $U$) | does knowledge help through the *output* side? |
| `rag_readout` | `rag` | $w_r + U e_r$ | are the two channels additive? |

**Why the readout matters for this task.** A positive sample for label $r$ is a patient whose *next* visit
contains $r$. Enriching the input embedding $z_r$ only helps through histories that already contain $r$
(recurrent codes); the label row $w_r$ is learned from the $n_r$ positives alone. The knowledge-conditioned
readout lets a rare label inherit a direction $U e_r$ learned from the frequent labels whose descriptions are
similar; this is the multi-label analogue of zero-shot label-description classifiers used in ICD coding.

Run on the cluster with `scripts/run_notebook.sbatch` (one configuration per job, `RAREPRED_RUN=name`), then
run the notebook once more with all names to aggregate (trained models are reloaded from `checkpoints/`).
Requires `processed/knowledge_embeddings_<encoder>.npz` from Notebook 5.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path(os.environ.get('RAREPRED_ROOT', '..')).resolve()
sys.path.insert(0, str(ROOT / 'src'))
PROCESSED   = Path(os.environ.get('RAREPRED_PROCESSED', ROOT / 'processed'))
EXTERNAL    = ROOT / 'data' / 'external'
RESULTS     = ROOT / 'results'
FIGURES     = ROOT / 'figures'
CHECKPOINTS = ROOT / 'checkpoints'
for d in (RESULTS, FIGURES, CHECKPOINTS):
    d.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd, torch
from rarepred.experiment import Setup, run_config
from rarepred.metrics import paired_bootstrap_delta
SEED       = int(os.environ.get('RAREPRED_SEED', 42))
MAX_EPOCHS = int(os.environ.get('RAREPRED_MAX_EPOCHS', 50))
RETRAIN    = os.environ.get('RAREPRED_RETRAIN', '0') == '1'
RUN        = [r for r in os.environ.get('RAREPRED_RUN', '').split(',') if r]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', DEVICE, '| seed', SEED, '| max epochs', MAX_EPOCHS)

ENCODER_TAG = os.environ.get('RAREPRED_ENCODER_TAG', 'BioLORD-2023')
RUN_DIR = 'knowledge_embeddings'
LAMBDA0 = float(os.environ.get('RAREPRED_LAMBDA0', 10.0))

import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 9, 'axes.titlesize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#666666', 'axes.labelcolor': '#191919', 'xtick.color': '#666666',
    'ytick.color': '#666666', 'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'legend.frameon': False,
})
BLUE, ORANGE, AQUA, GREY = '#257de0', '#f47230', '#00b894', '#9c9c9c'

## 1. Data, groups and the equivalence check

In [ ]:
S = Setup(PROCESSED, DEVICE, knowledge_npz=PROCESSED / f'knowledge_embeddings_{ENCODER_TAG}.npz',
          max_train=int(os.environ['RAREPRED_MAX_TRAIN']) if os.environ.get('RAREPRED_MAX_TRAIN') else None)
print(f"labels {S.D_out} | input vocab dx/rx/px {S.C} | train/test samples {len(S.train_samples):,}/{len(S.test_samples):,}")
print('sample-level quartile boundaries (train):', [int(b) for b in S.bounds_sample])
print('knowledge embeddings:', 'loaded (' + str(S.knowledge['encoder']) + ')' if S.knowledge else 'MISSING')
print(f"max |logit diff| default backbone vs Notebook 2 SharedBackbone: {S.equivalence_check():.2e}")

## 2. Train / load the configurations

In [ ]:
CONFIGS = {
    'baseline':    dict(emb='random'),
    'desc':        dict(emb='desc'),
    'rag':         dict(emb='rag'),
    'rag_shrink':  dict(emb='rag', lambda0=LAMBDA0),
    'readout':     dict(emb='random', readout='knowledge', label_source='rag'),
    'rag_readout': dict(emb='rag', readout='knowledge', label_source='rag'),
}
names = RUN or list(CONFIGS)

def run_all(configs, run_names):
    outs, preds = {}, {}
    for name in run_names:
        out, probs, ys, _ = run_config(S, name, configs[name], CHECKPOINTS, RESULTS / RUN_DIR,
                                       max_epochs=MAX_EPOCHS, seed=SEED, retrain=RETRAIN)
        outs[name], preds[name] = out, probs
    return outs, preds, ys

def summary_table(outs):
    rows = []
    for name, o in outs.items():
        r, p, mg = o['test_sample_quartiles'], o['test_patient_strata'], o['test_medco_mass_groups']
        rows.append({'config': name,
                     'sample-AUPRC': r['sample_auprc'],
                     'Q1 (rarest)': r['sample_auprc[Q1 (rarest)]'], 'Q2': r['sample_auprc[Q2]'],
                     'Q3': r['sample_auprc[Q3]'], 'Q4 (common)': r['sample_auprc[Q4 (common)]'],
                     'gap Q4-Q1': r['gap_sample_auprc'],
                     'label-AUPRC rare': p['label_auprc[rare]'], 'label-AUPRC common': p['label_auprc[common]'],
                     'MedCo-groups 0-25%': mg['sample_auprc[Q1 (rarest)]'],
                     'MedCo-groups 75-100%': mg['sample_auprc[Q4 (common)]'],
                     'acc@m': r[f"acc@{S.meta['m_median']}"]})
    return pd.DataFrame(rows).set_index('config').round(4)

outs, preds, ys = run_all(CONFIGS, names)
table = summary_table(outs)
table

## 3. Paired comparison with the baseline on the rarest quartile

Differences between configurations are evaluated on the *same* test samples (paired bootstrap over samples,
1,000 resamples). With about 700 test samples in Q1, differences below one AUPRC point are within noise.

In [ ]:
q1 = S.groups_sample['Q1 (rarest)']
if 'baseline' in preds:
    rows = []
    for name in preds:
        if name == 'baseline':
            continue
        d, lo, hi = paired_bootstrap_delta(preds['baseline'], preds[name], ys, q1)
        da, loa, hia = paired_bootstrap_delta(preds['baseline'], preds[name], ys)
        rows.append({'config': name, 'delta Q1': d, 'CI low': lo, 'CI high': hi,
                     'delta overall': da, 'CI low (overall)': loa, 'CI high (overall)': hia})
    deltas = pd.DataFrame(rows).set_index('config').round(4)
    display(deltas)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
qs = ['Q1 (rarest)', 'Q2', 'Q3', 'Q4 (common)']
x = np.arange(len(qs)); w = 0.8 / max(1, len(table))
colors = ['#9c9c9c', '#257de0', '#00b894', '#f2a600', '#6a4ce2', '#f47230']
for i, (name, row) in enumerate(table.iterrows()):
    ax.bar(x + (i - (len(table) - 1) / 2) * w, row[qs].values, width=w * 0.92, label=name, color=colors[i % 6])
ax.set_xticks(x, qs); ax.set_ylabel('test sample-AUPRC'); ax.legend(ncol=3, fontsize=8)
ax.set_title('Sample-AUPRC by label-frequency quartile', loc='left')
plt.tight_layout(); plt.savefig(FIGURES / 'knowledge_embeddings_quartiles.png', bbox_inches='tight'); plt.show()

## 4. Save

In [ ]:
with open(RESULTS / 'knowledge_embeddings_summary.json', 'w') as f:
    json.dump({'encoder': ENCODER_TAG, 'seed': SEED, 'table': table.reset_index().to_dict(orient='records'),
               'deltas_vs_baseline': (deltas.reset_index().to_dict(orient='records') if 'baseline' in preds else None)},
              f, indent=2)
print('saved results/knowledge_embeddings_summary.json')

## Reading the results

| observation | interpretation |
|---|---|
| `desc` > `baseline` on Q1 only | text priors fix poorly estimated rare input embeddings (Hypothesis A) |
| `rag` > `desc` | retrieved rare-disease knowledge adds information that the code description lacks |
| `rag_shrink` > `rag` | rare codes drift away from their prior during training; keeping them close helps |
| `readout` > `baseline` while `desc`/`rag` do not move Q1 | the bottleneck is on the label side (Hypothesis C), consistent with the new-diagnosis argument |
| no configuration closes the gap | knowledge is not the limiting factor at CCS granularity (see Notebook 4: rare CCS labels are mixtures of many diseases) |